In [269]:
import numpy as np

In [270]:
UP, RIGHT, DOWN, LEFT = 0, 1, 2, 3
DIRS = {UP: (-1, 0), RIGHT: (0, 1), DOWN: (1, 0), LEFT: (0, -1)}
ARROWS = {UP: "↑", RIGHT: "→", DOWN: "↓", LEFT: "←"}

In [271]:
class GridWorld:

    def __init__(
        self,
        rows: int,
        cols: int,
        step_reward: float,
        terminals: dict[tuple[int, int], float],
        walls: set[tuple[int, int]],
    ):
        self.rows = rows
        self.cols = cols
        self.step_reward = step_reward
        self.terminals = terminals
        self.walls = walls
        self.s2c = [
            (r, c)
            for r in range(self.rows)
            for c in range(self.cols)
            if (r, c) not in self.walls
        ]
        self.nS = len(self.s2c)
        self.nA = 4
        self.c2s = {cell: i for i, cell in enumerate(self.s2c)}

    def step(self, s: int, a: int):
        dr, dc = DIRS[a]
        cell = self.s2c[s]
        nxt = (cell[0] + dr, cell[1] + dc)
        if (
            not 0 <= nxt[0] < self.rows
            or not 0 <= nxt[1] < self.cols
            or nxt in self.walls
        ):
            nxt = cell
        reward = self.terminals.get(nxt, self.step_reward)
        return self.c2s[nxt], reward
    
    def cell_repr(self, r, c):
        cell = (r, c)
        if cell in self.terminals:
            return self.terminals[cell]
        elif cell in self.walls:
            return '#'
        else:
            return '·'
        
    def render(self):
        for r in range(self.rows):
            for c in range(self.cols):
                if r == 0 and c == 0:
                    print(" r/c", end="")
                    print(''.join([f'{v:>3} ' for v in range(self.cols)]))
                if c == 0:
                    print(f'{r:>3} ', end="")
                print(f'{self.cell_repr(r, c):>3} ', end="")
            print()
            
    def __repr__(self):
        return f'''
Grid(
    rows={self.rows},
    cols={self.cols},
    step_reward={self.step_reward},
    terminals={self.terminals},
    walls={self.walls},
)
'''
env = GridWorld(
    rows=3,
    cols=4,
    step_reward=0,
    terminals={(0, 3): 1},
    walls={(1, 1)},
)
env


Grid(
    rows=3,
    cols=4,
    step_reward=0,
    terminals={(0, 3): 1},
    walls={(1, 1)},
)

In [272]:
env.render()

 r/c  0   1   2   3 
  0   ·   ·   ·   1 
  1   ·   #   ·   · 
  2   ·   ·   ·   · 


In [273]:
def show_V(env: GridWorld, V: np.ndarray):
    for r in range(env.rows):
        for c in range(env.cols):
            if r == 0 and c == 0:
                print("  r/c", end="")
                print(''.join([f'{v:>4} ' for v in range(env.cols)]))
            if c == 0:
                print(f'{r:>4} ', end="")            
            cell = (r, c)
            if cell in env.c2s:
                value = round(V[env.c2s[(r, c)]], 2)
            else:
                value = '#'
            print(f'{value:>4} ', end="")
        print()

def q_from_v(
    env: GridWorld,
    V: np.ndarray,
    s: int,
    gamma: float,
):
    q = np.zeros(env.nA)
    if env.s2c[s] in env.terminals:
        return q
    for a in range(env.nA):
        ns, r = env.step(s, a)
        q[a] = r + gamma * V[ns]
    return q

def value_iteration(
    env: GridWorld,
    gamma=0.9,
    theta=1e-6,
    max_iters=1000,
    verbose=0,
):
    V = np.zeros(env.nS)
    if verbose >= 2:
        print('V init')
        show_V(env, V)
        print('-' * 25)    
    delta = float('inf')
    i = 0
    while delta >= theta and i < max_iters:
        delta = 0.0
        V_old = V.copy()
        for s in range(env.nS):
            V[s] = np.max(q_from_v(env, V_old, s, gamma))
            delta = max(delta, abs(V[s] - V_old[s]))
        if verbose >= 1:
            print(f"iter {i}: delta={delta:.6f}")
        if verbose >= 2:
            show_V(env, V)
            print('-' * 25)
        i += 1
    converged = delta < theta
    if converged:
        if verbose >= 1:
            print(f'value_iteration converged in {i - 1} iterations')
    else:
        print(f"value_iteration did not converge in {max_iters} iterations")
    return V, converged

In [274]:
V, converged = value_iteration(env)
V

array([0.81  , 0.9   , 1.    , 0.    , 0.729 , 0.9   , 1.    , 0.6561,
       0.729 , 0.81  , 0.9   ])

In [275]:
show_V(env, V)

  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73    #  0.9  1.0 
   2 0.66 0.73 0.81  0.9 


In [276]:
value_iteration(env, verbose=2);

V init
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 0: delta=1.000000
  r/c   0    1    2    3 
   0  0.0  0.0  1.0  0.0 
   1  0.0    #  0.0  1.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 1: delta=0.900000
  r/c   0    1    2    3 
   0  0.0  0.9  1.0  0.0 
   1  0.0    #  0.9  1.0 
   2  0.0  0.0  0.0  0.9 
-------------------------
iter 2: delta=0.810000
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1  0.0    #  0.9  1.0 
   2  0.0  0.0 0.81  0.9 
-------------------------
iter 3: delta=0.729000
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73    #  0.9  1.0 
   2  0.0 0.73 0.81  0.9 
-------------------------
iter 4: delta=0.656100
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73    #  0.9  1.0 
   2 0.66 0.73 0.81  0.9 
-------------------------
iter 5: delta=0.000000
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73    #  0.9  1.